# Validation extensions

This notebook adds hierarchical uncertainty estimates, controlled feature/event-weighting ablations, and reference-definition sensitivity analyses to the canonical experiment.

It loads the original implementation directly from `01_main_analysis.ipynb` and requires the authorized local outputs of that notebook. **Cell order in the main notebook is part of this dependency and must not be changed.** No execution outputs are included in this release.

In [ ]:
# CELL 1 — Imports, paths, original source, and frozen artifacts
import ast, copy, gc, hashlib, json, math, os, random, time, warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

from sklearn.preprocessing import SplineTransformer, StandardScaler, RobustScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, matthews_corrcoef,
    roc_auc_score, average_precision_score,
)

import torch
import torch.nn as nn
from IPython.display import display

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = DEVICE.type == 'cuda'
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision('high')
    except Exception:
        pass

TASK_DIR = Path(os.environ.get('TASK1_WORKDIR', '.')).expanduser().resolve()
_original_name = '01_main_analysis.ipynb'
_canonical_name = 'results'
ORIGINAL_NOTEBOOK = TASK_DIR / 'notebooks' / _original_name
CANONICAL_RESULT_DIR = TASK_DIR / _canonical_name
VALIDATION_DIR = TASK_DIR / 'validation_results'
BOOTSTRAP_DIR = VALIDATION_DIR / 'bootstrap'
ABLATION_DIR = VALIDATION_DIR / 'ablations'
SENSITIVITY_DIR = VALIDATION_DIR / 'reference_sensitivity'
AUDIT_DIR = VALIDATION_DIR / 'fidelity_audit'
for p in [VALIDATION_DIR, BOOTSTRAP_DIR, ABLATION_DIR, SENSITIVITY_DIR, AUDIT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

assert ORIGINAL_NOTEBOOK.exists(), f'Missing original notebook: {ORIGINAL_NOTEBOOK}'
assert CANONICAL_RESULT_DIR.exists(), f'Missing canonical results: {CANONICAL_RESULT_DIR}'

with open(ORIGINAL_NOTEBOOK, 'r', encoding='utf-8') as f:
    ORIGINAL_NB = json.load(f)

# Exact source cells used by the extension.
SRC_CONFIG = ''.join(ORIGINAL_NB['cells'][1]['source'])
SRC_CONSTANTS = ''.join(ORIGINAL_NB['cells'][2]['source'])
SRC_WEATHER_REFERENCE = ''.join(ORIGINAL_NB['cells'][4]['source'])
SRC_STATE = ''.join(ORIGINAL_NB['cells'][5]['source'])
SRC_DATA_LOAD = ''.join(ORIGINAL_NB['cells'][6]['source'])
SRC_TRUTH = ''.join(ORIGINAL_NB['cells'][7]['source'])
SRC_EVENTS = ''.join(ORIGINAL_NB['cells'][8]['source'])
SRC_FEATURES = ''.join(ORIGINAL_NB['cells'][9]['source'])
SRC_WEIGHTS = ''.join(ORIGINAL_NB['cells'][10]['source'])
SRC_TRAIN_CORE = ''.join(ORIGINAL_NB['cells'][12]['source'])
SRC_TRAIN_STAGE5 = ''.join(ORIGINAL_NB['cells'][13]['source'])
SRC_EVAL_UTILS = ''.join(ORIGINAL_NB['cells'][16]['source'])
SRC_HELDOUT_INFERENCE = ''.join(ORIGINAL_NB['cells'][17]['source'])

# Extract the exact original Config dataclass rather than maintaining a copied config.
_tree = ast.parse(SRC_CONFIG)
_cfg_node = next(n for n in _tree.body if isinstance(n, ast.ClassDef) and n.name == 'Config')
_cfg_source = '@dataclass\n' + ast.get_source_segment(SRC_CONFIG, _cfg_node)
_cfg_ns = {'dataclass': dataclass, 'Optional': Optional, 'os': os}
exec(_cfg_source, _cfg_ns)
OriginalConfig = _cfg_ns['Config']
CFG_BASE = OriginalConfig()

# Core canonical locations.
TRUTH_DIR = CANONICAL_RESULT_DIR / 'truth'
FEATURE_DIR = CANONICAL_RESULT_DIR / 'features'
PRED_DIR_CANONICAL = CANONICAL_RESULT_DIR / 'predictions'
TABLE_DIR_CANONICAL = CANONICAL_RESULT_DIR / 'tables'
MODEL_DIR_CANONICAL = CANONICAL_RESULT_DIR / 'models'

with open(CANONICAL_RESULT_DIR / 'Run_Manifest_FINAL_PAPER.json', 'r', encoding='utf-8') as f:
    RUN_MANIFEST = json.load(f)

BUILDING_NAMES = list(RUN_MANIFEST['building_set'])
CANONICAL_SEEDS = tuple(RUN_MANIFEST['ensemble_seeds'])
FEATURE_NAMES_FULL = list(RUN_MANIFEST['feature_names'])
DECISION_THRESHOLD = float(RUN_MANIFEST['decoder']['decision_threshold'])

assert BUILDING_NAMES == [f'S{i}' for i in range(1, 8)]
assert CANONICAL_SEEDS == (42,123,456,789,2026)
assert len(FEATURE_NAMES_FULL) == 41
assert DECISION_THRESHOLD == 0.50

# Load frozen truth, features, predictions and paper tables.
TRUTH = {}
FEATURE_TABLES = {}
INTERVAL_CANONICAL = {}
for b in BUILDING_NAMES:
    work = pd.read_pickle(TRUTH_DIR / f'{b}_truth.pkl.gz', compression='gzip')
    with open(TRUTH_DIR / f'{b}_truth_summary.json', 'r', encoding='utf-8') as f:
        summary = json.load(f)
    TRUTH[b] = {'work': work, 'summary': summary}

    feat = pd.read_pickle(FEATURE_DIR / f'{b}_context_features.pkl.gz', compression='gzip')
    FEATURE_TABLES[b] = feat

    pred = pd.read_csv(PRED_DIR_CANONICAL / f'{b}_interval_predictions.csv', parse_dates=['DateTime_local'])
    INTERVAL_CANONICAL[b] = pred.set_index('DateTime_local').sort_index()

DAILY_CANONICAL = pd.read_csv(TABLE_DIR_CANONICAL / 'daily_operational_schedule.csv')
for c in ['date','true_turn_on','true_turn_off','pred_turn_on','pred_turn_off']:
    DAILY_CANONICAL[c] = pd.to_datetime(DAILY_CANONICAL[c], errors='coerce')
POINTWISE_CANONICAL = pd.read_csv(TABLE_DIR_CANONICAL / 'pointwise_metrics_by_building.csv')

# Original scalar/grid names are initialized exactly as in CELL 1 of the frozen notebook.
FREQ_MIN = CFG_BASE.dt_minutes
EXPECTED_DAILY_SAMPLES = CFG_BASE.steps_per_day
DATETIME_COL = 'DateTime_local'
WEATHER_DATETIME_COL = 'DateTime_local'
TEMPERATURE_COL = 'outdoor_temperature'
DIRECT_RADIATION_COL = 'direct_radiation'
SHORTWAVE_RADIATION_COL = 'shortwave_radiation'
DIRECT_RADIATION_COMPLETENESS_THRESHOLD = 0.95
SEASONS = {}

BOOTSTRAP_REPS = 5000
BOOTSTRAP_SEED = 20261005
CI_LEVEL = 0.95

print('Task directory:', TASK_DIR)
print('Frozen canonical result directory:', CANONICAL_RESULT_DIR)
print('Validation output directory:', VALIDATION_DIR)
print('Device:', DEVICE)

## Canonical-run consistency

The source and artifact audit checks that validation experiments use the original configuration, preprocessing, checkpoints, and evaluation definitions. Ablation variants reuse the same eligible data rows.

In [ ]:
# CELL 2 — Exact configuration, checkpoint, preprocessing, and source-code audit
# Execute the original constants cell to obtain BUILDINGS/reference settings exactly.
CFG = CFG_BASE
exec(SRC_CONSTANTS, globals())
assert BUILDING_NAMES == list(RUN_MANIFEST['building_set'])

# Re-execute the exact original weather-reference and raw-data loading cells.
# This is required because the original held-out monthly evaluator calls month_qc(),
# which depends on load_raw and the original resolved SEASONS. No alternative season
# reconstruction is used in this extension.
LOAD_FILE = TASK_DIR / CFG_BASE.load_file
WEATHER_FILE = TASK_DIR / CFG_BASE.weather_file
exec(SRC_WEATHER_REFERENCE, globals())
exec(SRC_DATA_LOAD, globals())

# Extract the exact original season_mask() helper without executing the truth-cache loop.
_truth_tree = ast.parse(SRC_TRUTH)
_season_node = next(n for n in _truth_tree.body if isinstance(n, ast.FunctionDef) and n.name == 'season_mask')
exec(ast.get_source_segment(SRC_TRUTH, _season_node), globals())

# Season endpoints must agree with every frozen truth index.
for b in BUILDING_NAMES:
    idx = TRUTH[b]['work'].index
    expected = idx[season_mask(idx)]
    assert len(expected) == len(idx), (b, len(expected), len(idx))
    for _, (s, e) in SEASONS.items():
        part = idx[(idx >= pd.Timestamp(s)) & (idx <= pd.Timestamp(e))]
        if len(part):
            assert part.min() == pd.Timestamp(s)
            assert part.max() == pd.Timestamp(e)

# Feature groups in the exact order used by the final notebook.
LOAD_FEATURES = [
    'p_h_score','p_h_z','p_h_med_rel','p_h_iqr_rel',
    'p_v_score','p_v_z','p_v_med_rel','p_v_iqr_rel',
    'p_g_score','p_g_z','p_g_riqr',
]
TEMP_FEATURES = [
    't_h_score','t_h_z','t_h_med','t_h_iqr',
    't_v_score','t_v_z','t_v_med','t_v_iqr',
    't_g_score','t_g_z','t_g_med','t_g_iqr',
]
RAD_FEATURES = [
    'r_h_score','r_h_z','r_h_med','r_h_iqr',
    'r_v_score','r_v_z','r_v_med','r_v_iqr',
    'r_g_score','r_g_z','r_g_med','r_g_iqr',
]
DELTA_FEATURES = ['p_delta15','p_delta30','p_delta60']
CALENDAR_FEATURES = ['tod_sin','tod_cos','is_weekend']
assert FEATURE_NAMES_FULL == LOAD_FEATURES + TEMP_FEATURES + RAD_FEATURES + DELTA_FEATURES + CALENDAR_FEATURES

# Check the exact original config against all 35 saved canonical seed checkpoints.
config_reference = asdict(CFG_BASE)
checkpoint_audit = []
for b in BUILDING_NAMES:
    for seed in CANONICAL_SEEDS:
        p = MODEL_DIR_CANONICAL / b / f'seed_{seed}_best.pt'
        payload = torch.load(p, map_location='cpu', weights_only=False)
        i = BUILDING_NAMES.index(b)
        expected_val = BUILDING_NAMES[(i + 1) % len(BUILDING_NAMES)]
        expected_train = [x for x in BUILDING_NAMES if x not in {b, expected_val}]
        checkpoint_audit.append({
            'building': b,
            'seed': seed,
            'config_exact': payload['config'] == config_reference,
            'feature_order_exact': list(payload['feature_names']) == FEATURE_NAMES_FULL,
            'building_set_exact': list(payload['building_set']) == BUILDING_NAMES,
            'best_stage_is_5': int(payload['best_stage']) == len(CFG_BASE.curriculum),
            'all_stages_reached': bool(payload['all_stages_reached']),
            'test_building_exact': payload['test_building'] == b,
            'validation_building_exact': payload['val_building'] == expected_val,
            'training_buildings_exact': list(payload['train_buildings']) == expected_train,
            'training_logic_version_exact': payload['training_logic_version'] == 'event_aware_daily_schedule_stage5_eventrulefix_v3',
        })
checkpoint_audit = pd.DataFrame(checkpoint_audit)
assert checkpoint_audit.drop(columns=['building','seed']).all().all(), checkpoint_audit
checkpoint_audit.to_csv(AUDIT_DIR / 'Canonical_Checkpoint_Audit.csv', index=False)

# -----------------------------------------------------------------------------
# Raw-input parity audit.
# The ablations consume the frozen original feature tables, but the original
# monthly evaluator still reads load_raw/SEASONS. Fail immediately if the
# current Excel inputs no longer match the data embedded in the frozen truth.
# -----------------------------------------------------------------------------
def _numeric_series_exact(a, b, atol=1e-12):
    aa = pd.to_numeric(a, errors='coerce').to_numpy(float)
    bb = pd.to_numeric(b, errors='coerce').to_numpy(float)
    return (
        len(aa) == len(bb)
        and np.array_equal(np.isnan(aa), np.isnan(bb))
        and np.allclose(aa, bb, rtol=0, atol=atol, equal_nan=True)
    )

raw_parity_rows = []
for b in BUILDING_NAMES:
    cfg_b = BUILDINGS[b]
    frozen = TRUTH[b]['work']
    idx = frozen.index
    total_ok = _numeric_series_exact(
        frozen[cfg_b['total']],
        pd.to_numeric(load_raw[cfg_b['total']], errors='coerce').reindex(idx),
    )
    chiller_ok = _numeric_series_exact(
        frozen[cfg_b['chiller']],
        pd.to_numeric(load_raw[cfg_b['chiller']], errors='coerce').reindex(idx),
    )
    weather_checks = {}
    for c in [TEMPERATURE_COL, RADIATION_COL] + REFERENCE_FEATURES:
        weather_checks[c] = _numeric_series_exact(
            frozen[c],
            pd.to_numeric(weather_features[c], errors='coerce').reindex(idx),
        )
    raw_parity_rows.append({
        'building': b,
        'total_load_exact': total_ok,
        'chiller_power_exact': chiller_ok,
        'weather_and_sttr_inputs_exact': bool(all(weather_checks.values())),
    })
RAW_INPUT_PARITY = pd.DataFrame(raw_parity_rows)
assert RAW_INPUT_PARITY.drop(columns='building').all().all(), RAW_INPUT_PARITY
RAW_INPUT_PARITY.to_csv(AUDIT_DIR / 'Raw_Input_Parity_Against_Frozen_Truth.csv', index=False)

# Frozen artifact fingerprints are part of the ablation checkpoint identity.
def _sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            block = f.read(chunk)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

_fingerprint_rows = []
for b in BUILDING_NAMES:
    for kind, path in [
        ('truth', TRUTH_DIR / f'{b}_truth.pkl.gz'),
        ('features', FEATURE_DIR / f'{b}_context_features.pkl.gz'),
    ]:
        _fingerprint_rows.append({
            'building': b,
            'kind': kind,
            'path': str(path),
            'sha256': _sha256_file(path),
        })
FROZEN_ARTIFACT_FINGERPRINTS = pd.DataFrame(_fingerprint_rows)
FROZEN_ARTIFACT_FINGERPRINTS.to_csv(AUDIT_DIR / 'Frozen_Artifact_SHA256.csv', index=False)
FROZEN_DATA_SIGNATURE = hashlib.sha256(
    '\n'.join(
        f"{r.building}|{r.kind}|{r.sha256}"
        for r in FROZEN_ARTIFACT_FINGERPRINTS.itertuples(index=False)
    ).encode('utf-8')
).hexdigest()

# Cached preprocessing invariants.
preprocess_rows = []
for b in BUILDING_NAMES:
    feat = FEATURE_TABLES[b]
    truth = pd.to_numeric(TRUTH[b]['work']['binary_on'], errors='coerce').reindex(feat.index)
    ft = pd.to_numeric(feat['true_state'], errors='coerce')
    same_target = np.array_equal(ft.isna().to_numpy(), truth.isna().to_numpy()) and np.array_equal(
        ft.dropna().to_numpy(float), truth.dropna().to_numpy(float)
    )
    feature_finite = np.isfinite(feat[FEATURE_NAMES_FULL].to_numpy(float)).all()
    preprocess_rows.append({
        'building': b,
        'rows': len(feat),
        'feature_count': len(FEATURE_NAMES_FULL),
        'all_41_inputs_finite': bool(feature_finite),
        'cached_true_state_exactly_matches_frozen_reference': bool(same_target),
        'index_unique': bool(feat.index.is_unique),
        'index_monotonic': bool(feat.index.is_monotonic_increasing),
    })
preprocess_audit = pd.DataFrame(preprocess_rows)
assert preprocess_audit[['all_41_inputs_finite','cached_true_state_exactly_matches_frozen_reference','index_unique','index_monotonic']].all().all()
preprocess_audit.to_csv(AUDIT_DIR / 'Frozen_Preprocessing_Audit.csv', index=False)

# -----------------------------------------------------------------------------
# Canonical training-history audit: every saved original seed must have followed
# the exact five-stage learning-rate/event-weight curriculum in Config.
# -----------------------------------------------------------------------------
_HISTORY_WEIGHT_COLS = [
    'w_ordinary','w_secondary_pm30','w_secondary_pm15','w_secondary_exact',
    'w_primary_pm30','w_primary_pm15','w_primary_exact',
]
canonical_history_rows = []
for b in BUILDING_NAMES:
    for seed in CANONICAL_SEEDS:
        hp = MODEL_DIR_CANONICAL / b / f'seed_{seed}_history.csv'
        hist = pd.read_csv(hp)
        stages_exact = set(pd.to_numeric(hist['stage']).astype(int).unique()) == set(range(1, len(CFG_BASE.curriculum)+1))
        schedule_exact = True
        min_stage_epochs_ok = True
        warmup_caps_ok = True
        for stage_no, row in enumerate(CFG_BASE.curriculum, start=1):
            hs = hist.loc[pd.to_numeric(hist['stage']).astype(int).eq(stage_no)]
            if hs.empty:
                schedule_exact = False
                min_stage_epochs_ok = False
                continue
            schedule_exact &= bool(np.allclose(pd.to_numeric(hs['lr']), float(row[0]), rtol=0, atol=1e-15))
            for c, v in zip(_HISTORY_WEIGHT_COLS, row[1:]):
                schedule_exact &= bool(np.allclose(pd.to_numeric(hs[c]), float(v), rtol=0, atol=1e-12))
            min_stage_epochs_ok &= int(pd.to_numeric(hs['epochs_in_stage']).max()) >= int(CFG_BASE.min_epochs_per_stage)
            if stage_no < len(CFG_BASE.curriculum):
                warmup_caps_ok &= int(pd.to_numeric(hs['epochs_in_stage']).max()) <= int(CFG_BASE.max_epochs_per_stage)
        total_epoch_ok = int(pd.to_numeric(hist['epoch']).max()) <= int(CFG_BASE.max_epochs)
        canonical_history_rows.append({
            'building': b,
            'seed': int(seed),
            'all_five_stages_present': bool(stages_exact),
            'lr_and_event_weight_schedule_exact': bool(schedule_exact),
            'minimum_stage_epochs_respected': bool(min_stage_epochs_ok),
            'warmup_stage_caps_respected': bool(warmup_caps_ok),
            'absolute_epoch_ceiling_respected': bool(total_epoch_ok),
        })
CANONICAL_HISTORY_AUDIT = pd.DataFrame(canonical_history_rows)
assert CANONICAL_HISTORY_AUDIT.drop(columns=['building','seed']).all().all(), CANONICAL_HISTORY_AUDIT
CANONICAL_HISTORY_AUDIT.to_csv(AUDIT_DIR / 'Canonical_Training_History_Schedule_Audit.csv', index=False)

# Source hashes make the exact original cells used by this extension explicit.
source_cells = {
    'config': SRC_CONFIG,
    'constants': SRC_CONSTANTS,
    'weather_reference': SRC_WEATHER_REFERENCE,
    'raw_data_load_and_seasons': SRC_DATA_LOAD,
    'state_assignment': SRC_STATE,
    'event_semantics': SRC_EVENTS,
    'feature_preprocessing': SRC_FEATURES,
    'event_weights': SRC_WEIGHTS,
    'training_core': SRC_TRAIN_CORE,
    'training_stage5': SRC_TRAIN_STAGE5,
    'evaluation_utils': SRC_EVAL_UTILS,
    'heldout_inference': SRC_HELDOUT_INFERENCE,
}
source_hashes = pd.DataFrame([
    {'component': k, 'sha256': hashlib.sha256(v.encode('utf-8')).hexdigest()}
    for k,v in source_cells.items()
])
source_hashes.to_csv(AUDIT_DIR / 'Original_Source_Cell_SHA256.csv', index=False)

TRAINING_DEPENDENCY_SIGNATURE = hashlib.sha256(
    (
        SRC_CONFIG + SRC_CONSTANTS + SRC_EVENTS + SRC_FEATURES
        + SRC_TRAIN_CORE + SRC_TRAIN_STAGE5
        + FROZEN_DATA_SIGNATURE
    ).encode('utf-8')
).hexdigest()

display(checkpoint_audit.head())
display(preprocess_audit)
print('PASS: raw inputs, canonical config/checkpoints, training histories, and frozen preprocessing artifacts are internally exact.')

In [ ]:
# CELL 3 — Load the exact original state, event, and evaluation semantics
# Exact original state assignment/smoothing.
exec(SRC_STATE, globals())

# Exact binary mapping from the original truth cell.
STATE_TO_BINARY = {
    'OFF': 0.0,
    'AUX': 0.0,
    'ACTIVE-PART': 1.0,
    'ACTIVE-HIGH': 1.0,
}

# Exact original operational-event code. This also rebuilds TRAINING_EVENT_META.
exec(SRC_EVENTS, globals())
assert set(TRAINING_EVENT_META) == set(BUILDING_NAMES)

# Verify cached primary/secondary event distances against the event metadata
# regenerated by the exact original event rules.
event_meta_audit = []
for b in BUILDING_NAMES:
    feat = FEATURE_TABLES[b]
    meta = TRAINING_EVENT_META[b]
    p_cached = pd.to_numeric(feat['primary_event_distance'], errors='coerce')
    s_cached = pd.to_numeric(feat['secondary_transition_distance'], errors='coerce')
    p_exact = pd.to_numeric(meta['primary_event_distance'], errors='coerce').reindex(feat.index)
    s_exact = pd.to_numeric(meta['secondary_transition_distance'], errors='coerce').reindex(feat.index)
    p_ok = np.array_equal(p_cached.to_numpy(), p_exact.to_numpy(), equal_nan=True)
    s_ok = np.array_equal(s_cached.to_numpy(), s_exact.to_numpy(), equal_nan=True)
    event_meta_audit.append({'building':b,'primary_event_distance_exact':p_ok,'secondary_transition_distance_exact':s_ok})
event_meta_audit = pd.DataFrame(event_meta_audit)
assert event_meta_audit.drop(columns='building').all().all(), event_meta_audit
event_meta_audit.to_csv(AUDIT_DIR / 'Event_Metadata_Audit.csv', index=False)

print('PASS: exact original state/event semantics reproduce the cached training-event metadata.')

## Hierarchical uncertainty estimation

Bootstrap replicates sample systems, then calendar days within each selected system. Day-level observations remain clustered so interval predictions, event labels, and timing errors are resampled together. Macro and pooled statistics are reported separately.

In [ ]:
# CELL 4 — Day-cluster sufficient statistics and metric engine

def _confusion_counts(y, p):
    y = np.asarray(y, int); p = np.asarray(p, int)
    return {
        'tn': int(np.sum((y == 0) & (p == 0))),
        'fp': int(np.sum((y == 0) & (p == 1))),
        'fn': int(np.sum((y == 1) & (p == 0))),
        'tp': int(np.sum((y == 1) & (p == 1))),
    }


def _f1_from_counts(tn, fp, fn, tp):
    den = 2*tp + fp + fn
    return 2*tp/den if den else np.nan


def _mcc_from_counts(tn, fp, fn, tp):
    den = math.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
    return (tp*tn - fp*fn)/den if den else np.nan


def _event_prf(true_exists, pred_exists):
    t = np.asarray(true_exists, bool); p = np.asarray(pred_exists, bool)
    tp = int(np.sum(t & p)); fp = int(np.sum(~t & p)); fn = int(np.sum(t & ~p))
    precision = tp/(tp+fp) if (tp+fp) else np.nan
    recall = tp/(tp+fn) if (tp+fn) else np.nan
    f1 = 2*precision*recall/(precision+recall) if np.isfinite(precision) and np.isfinite(recall) and (precision+recall)>0 else np.nan
    return {'precision':precision,'recall':recall,'f1':f1,'tp':tp,'fp':fp,'fn':fn}


def _timing_metrics(errors):
    e = pd.to_numeric(pd.Series(errors), errors='coerce').dropna().to_numpy(float)
    if len(e) == 0:
        return {k:np.nan for k in ['mae','median_ae','rmse','bias','q90_ae','within15','within30']} | {'matched_n':0}
    ae = np.abs(e)
    return {
        'matched_n': int(len(e)),
        'mae': float(np.mean(ae)),
        'median_ae': float(np.median(ae)),
        'rmse': float(np.sqrt(np.mean(e**2))),
        'bias': float(np.mean(e)),
        'q90_ae': float(np.quantile(ae, 0.90)),
        'within15': float(np.mean(ae <= CFG_BASE.timing_tolerance_15_min)),
        'within30': float(np.mean(ae <= CFG_BASE.timing_tolerance_30_min)),
    }


def build_day_cluster_tables(interval_by_building, daily_df):
    out = {}
    for b in BUILDING_NAMES:
        pred = interval_by_building[b].copy()
        pred['date'] = pred.index.normalize()
        valid = pred['true_state'].notna() & pred['predicted_state'].notna()
        interval_rows = []
        for date, g in pred.loc[valid].groupby('date'):
            cc = _confusion_counts(g['true_state'].astype(int), g['predicted_state'].astype(int))
            interval_rows.append({'date':pd.Timestamp(date), **cc, 'n_intervals':int(len(g))})
        interval_day = pd.DataFrame(interval_rows)

        daily = daily_df.loc[daily_df['building'].eq(b)].copy()
        daily['date'] = pd.to_datetime(daily['date']).dt.normalize()
        daily['on_true_exists'] = daily['true_turn_on'].notna()
        daily['on_pred_exists'] = daily['pred_turn_on'].notna()
        daily['off_true_exists'] = daily['true_turn_off'].notna()
        daily['off_pred_exists'] = daily['pred_turn_off'].notna()
        keep = [
            'date','daily_eval_eligible',
            'on_true_exists','on_pred_exists','off_true_exists','off_pred_exists',
            'turn_on_error_min','turn_off_error_min',
        ]
        merged = daily[keep].merge(interval_day, on='date', how='outer')
        for c in ['tn','fp','fn','tp','n_intervals']:
            merged[c] = pd.to_numeric(merged[c], errors='coerce').fillna(0).astype(int)
        merged['daily_eval_eligible'] = merged['daily_eval_eligible'].fillna(False).astype(bool)
        for c in ['on_true_exists','on_pred_exists','off_true_exists','off_pred_exists']:
            merged[c] = merged[c].fillna(False).astype(bool)
        out[b] = merged.sort_values('date').reset_index(drop=True)
    return out


def compute_metrics_from_sampled_clusters(sampled_clusters):
    per_system = []
    pooled_parts = []
    for df in sampled_clusters:
        df = df.copy(); pooled_parts.append(df)
        tn,fp,fn,tp = [int(df[c].sum()) for c in ['tn','fp','fn','tp']]
        row = {
            'state_f1': _f1_from_counts(tn,fp,fn,tp),
            'state_mcc': _mcc_from_counts(tn,fp,fn,tp),
        }
        elig = df['daily_eval_eligible'].to_numpy(bool)
        for ev in ['on','off']:
            det = _event_prf(df.loc[elig,f'{ev}_true_exists'], df.loc[elig,f'{ev}_pred_exists'])
            for k in ['precision','recall','f1']:
                row[f'{ev}_event_{k}'] = det[k]
            tm = _timing_metrics(df.loc[elig,f'turn_{ev}_error_min'])
            for k,v in tm.items(): row[f'{ev}_timing_{k}'] = v
        per_system.append(row)

    pooled = pd.concat(pooled_parts, ignore_index=True)
    tn,fp,fn,tp = [int(pooled[c].sum()) for c in ['tn','fp','fn','tp']]
    result = {
        'state_f1_pooled': _f1_from_counts(tn,fp,fn,tp),
        'state_mcc_pooled': _mcc_from_counts(tn,fp,fn,tp),
        'state_f1_macro': float(np.nanmean([r['state_f1'] for r in per_system])),
        'state_mcc_macro': float(np.nanmean([r['state_mcc'] for r in per_system])),
    }
    elig = pooled['daily_eval_eligible'].to_numpy(bool)
    for ev in ['on','off']:
        det = _event_prf(pooled.loc[elig,f'{ev}_true_exists'], pooled.loc[elig,f'{ev}_pred_exists'])
        for k in ['precision','recall','f1']:
            result[f'{ev}_event_{k}_pooled'] = det[k]
            result[f'{ev}_event_{k}_macro'] = float(np.nanmean([r[f'{ev}_event_{k}'] for r in per_system]))
        tm = _timing_metrics(pooled.loc[elig,f'turn_{ev}_error_min'])
        for k in ['mae','median_ae','rmse','bias','q90_ae','within15','within30']:
            result[f'{ev}_timing_{k}_pooled'] = tm[k]
            result[f'{ev}_timing_{k}_macro'] = float(np.nanmean([r[f'{ev}_timing_{k}'] for r in per_system]))
    return result


def hierarchical_bootstrap(cluster_tables, n_boot=5000, seed=20261005, ci=0.95):
    systems = list(BUILDING_NAMES)
    central = compute_metrics_from_sampled_clusters([cluster_tables[b] for b in systems])
    rng = np.random.default_rng(seed)
    reps = []
    for _ in range(int(n_boot)):
        sampled_systems = rng.choice(systems, size=len(systems), replace=True)
        copies = []
        for b in sampled_systems:
            src = cluster_tables[b]
            idx = rng.integers(0, len(src), size=len(src))
            copies.append(src.iloc[idx].reset_index(drop=True))
        reps.append(compute_metrics_from_sampled_clusters(copies))
    rep_df = pd.DataFrame(reps)
    alpha = (1-ci)/2
    rows = []
    for metric, estimate in central.items():
        vals = pd.to_numeric(rep_df[metric], errors='coerce').dropna()
        rows.append({
            'metric':metric,
            'estimate':float(estimate) if np.isfinite(estimate) else np.nan,
            'ci_level':ci,
            'ci_low':float(vals.quantile(alpha)) if len(vals) else np.nan,
            'ci_high':float(vals.quantile(1-alpha)) if len(vals) else np.nan,
            'bootstrap_replicates_valid':int(len(vals)),
            'bootstrap_scheme':'systems with replacement; calendar-day clusters with replacement within each sampled system copy',
        })
    return pd.DataFrame(rows), rep_df

In [ ]:
# CELL 5 — Canonical uncertainty intervals + macro/pooled audit
CANONICAL_CLUSTERS = build_day_cluster_tables(INTERVAL_CANONICAL, DAILY_CANONICAL)
canonical_central = compute_metrics_from_sampled_clusters([CANONICAL_CLUSTERS[b] for b in BUILDING_NAMES])

# Exact central-estimate parity checks against the frozen paper outputs.
# These assertions verify that the custom sufficient-statistic/bootstrap engine
# reproduces the original evaluation semantics before any resampling is performed.
assert np.isclose(canonical_central['state_f1_macro'], POINTWISE_CANONICAL['f1_on'].mean(), atol=1e-12)
assert np.isclose(canonical_central['state_mcc_macro'], POINTWISE_CANONICAL['mcc'].mean(), atol=1e-12)

TABLE3 = pd.read_csv(TABLE_DIR_CANONICAL / 'Table3_EventDetection.csv').set_index('Building')
TABLE4 = pd.read_csv(TABLE_DIR_CANONICAL / 'Table4_EventTiming.csv').set_index('Building')
TABLE2 = pd.read_csv(TABLE_DIR_CANONICAL / 'Table2_Pointwise.csv').set_index('Building')
parity_rows = []
for b in BUILDING_NAMES:
    df = CANONICAL_CLUSTERS[b]
    tn,fp,fn,tp = [int(df[c].sum()) for c in ['tn','fp','fn','tp']]
    f1b = _f1_from_counts(tn,fp,fn,tp)
    mccb = _mcc_from_counts(tn,fp,fn,tp)
    assert np.isclose(f1b, TABLE2.loc[b,'ON F1'], atol=1e-12)
    assert np.isclose(mccb, TABLE2.loc[b,'MCC'], atol=1e-12)
    elig = df['daily_eval_eligible'].to_numpy(bool)
    for ev, label in [('on','ON'),('off','OFF')]:
        det = _event_prf(df.loc[elig,f'{ev}_true_exists'], df.loc[elig,f'{ev}_pred_exists'])
        tm = _timing_metrics(df.loc[elig,f'turn_{ev}_error_min'])
        assert np.isclose(det['precision'], TABLE3.loc[b,f'{label} precision'], atol=1e-12, equal_nan=True)
        assert np.isclose(det['recall'], TABLE3.loc[b,f'{label} recall'], atol=1e-12, equal_nan=True)
        assert np.isclose(det['f1'], TABLE3.loc[b,f'{label} F1'], atol=1e-12, equal_nan=True)
        assert np.isclose(tm['mae'], TABLE4.loc[b,f'{label} MAE [min]'], atol=1e-12, equal_nan=True)
        assert np.isclose(tm['median_ae'], TABLE4.loc[b,f'{label} median AE [min]'], atol=1e-12, equal_nan=True)
        assert np.isclose(tm['rmse'], TABLE4.loc[b,f'{label} RMSE [min]'], atol=1e-12, equal_nan=True)
        assert np.isclose(tm['bias'], TABLE4.loc[b,f'{label} bias [min]'], atol=1e-12, equal_nan=True)
        assert np.isclose(tm['q90_ae'], TABLE4.loc[b,f'{label} q90 AE [min]'], atol=1e-12, equal_nan=True)
        assert np.isclose(tm['within15'], TABLE4.loc[b,f'{label} within ±15'], atol=1e-12, equal_nan=True)
        assert np.isclose(tm['within30'], TABLE4.loc[b,f'{label} within ±30'], atol=1e-12, equal_nan=True)
    parity_rows.append({'building':b,'pointwise_exact':True,'event_detection_exact':True,'timing_exact':True})

GLOBAL = pd.read_csv(TABLE_DIR_CANONICAL / 'Table_Global_Summary.csv').set_index('Metric')['Value']
global_map = {
    'on_event_precision_pooled':'Pooled turn-on precision',
    'on_event_recall_pooled':'Pooled turn-on recall',
    'on_event_f1_pooled':'Pooled turn-on detection F1',
    'on_timing_mae_pooled':'Pooled turn-on timing MAE [min]',
    'on_timing_median_ae_pooled':'Pooled turn-on median AE [min]',
    'on_timing_bias_pooled':'Pooled turn-on bias [min]',
    'on_timing_within15_pooled':'Pooled turn-on within ±15 min',
    'on_timing_within30_pooled':'Pooled turn-on within ±30 min',
    'off_event_precision_pooled':'Pooled turn-off precision',
    'off_event_recall_pooled':'Pooled turn-off recall',
    'off_event_f1_pooled':'Pooled turn-off detection F1',
    'off_timing_mae_pooled':'Pooled turn-off timing MAE [min]',
    'off_timing_median_ae_pooled':'Pooled turn-off median AE [min]',
    'off_timing_bias_pooled':'Pooled turn-off bias [min]',
    'off_timing_within15_pooled':'Pooled turn-off within ±15 min',
    'off_timing_within30_pooled':'Pooled turn-off within ±30 min',
}
for k,label in global_map.items():
    assert np.isclose(canonical_central[k], float(GLOBAL.loc[label]), atol=1e-12, equal_nan=True), (k,canonical_central[k],GLOBAL.loc[label])

CANONICAL_METRIC_PARITY = pd.DataFrame(parity_rows)
CANONICAL_METRIC_PARITY.to_csv(AUDIT_DIR / 'Canonical_Metric_Engine_Parity.csv', index=False)
print('PASS: bootstrap metric engine reproduces all frozen per-system event/timing metrics and pooled headline metrics exactly.')

bootstrap_summary, bootstrap_replicates = hierarchical_bootstrap(
    CANONICAL_CLUSTERS,
    n_boot=BOOTSTRAP_REPS,
    seed=BOOTSTRAP_SEED,
    ci=CI_LEVEL,
)
bootstrap_summary.to_csv(BOOTSTRAP_DIR / 'Bootstrap_Primary_Metrics_Macro_Pooled_95CI.csv', index=False)
bootstrap_replicates.to_csv(BOOTSTRAP_DIR / 'Bootstrap_Replicates.csv.gz', index=False, compression='gzip')

# Compact headline table requested for the manuscript.
headline_metrics = [
    'state_f1_macro','state_mcc_macro',
    'on_event_f1_macro','on_event_f1_pooled','off_event_f1_macro','off_event_f1_pooled',
    'on_timing_mae_macro','on_timing_mae_pooled','off_timing_mae_macro','off_timing_mae_pooled',
    'on_timing_rmse_macro','on_timing_rmse_pooled','off_timing_rmse_macro','off_timing_rmse_pooled',
]
headline_ci = bootstrap_summary.loc[bootstrap_summary['metric'].isin(headline_metrics)].copy()
headline_ci.to_csv(BOOTSTRAP_DIR / 'Bootstrap_Headline_Metrics_95CI.csv', index=False)
display(headline_ci)

## Controlled ablations

The comparisons assess load-only features, load plus weather, full H/V/G features, and the full feature set without event weighting. All variants retain the canonical data-eligibility rules, training schedule, and held-out evaluation protocol.

In [ ]:
# CELL 6 — Variant definitions and exact-original training/evaluation namespace
FEATURE_SETS = {
    'load_only_eventaware': LOAD_FEATURES + DELTA_FEATURES,
    'load_weather_eventaware': LOAD_FEATURES + TEMP_FEATURES + RAD_FEATURES + DELTA_FEATURES,
    'full_hvg_eventaware': FEATURE_NAMES_FULL,
    'full_no_event_weighting': FEATURE_NAMES_FULL,
}
assert len(FEATURE_SETS['load_only_eventaware']) == 14
assert len(FEATURE_SETS['load_weather_eventaware']) == 38
assert len(FEATURE_SETS['full_hvg_eventaware']) == 41
assert len(FEATURE_SETS['full_no_event_weighting']) == 41


def _cfg_for_variant(variant):
    c = copy.deepcopy(CFG_BASE)
    if variant == 'full_no_event_weighting':
        # Same five LRs and same curriculum/stopping policy; only weights become 1.
        c.curriculum = tuple((row[0],1.0,1.0,1.0,1.0,1.0,1.0,1.0) for row in c.curriculum)
        c.fixed_weight_ordinary = 1.0
        c.fixed_weight_secondary_pm30 = 1.0
        c.fixed_weight_secondary_pm15 = 1.0
        c.fixed_weight_secondary_exact = 1.0
        c.fixed_weight_primary_pm30 = 1.0
        c.fixed_weight_primary_pm15 = 1.0
        c.fixed_weight_primary_exact = 1.0
    return c


def _fixed_weights_from_cfg(c):
    return (
        c.fixed_weight_ordinary,
        c.fixed_weight_secondary_pm30,
        c.fixed_weight_secondary_pm15,
        c.fixed_weight_secondary_exact,
        c.fixed_weight_primary_pm30,
        c.fixed_weight_primary_pm15,
        c.fixed_weight_primary_exact,
    )


def make_exact_variant_namespace(variant):
    if variant == 'full_hvg_eventaware':
        raise ValueError('The canonical full model is loaded from the frozen paper run and is not retrained.')

    variant_dir = ABLATION_DIR / variant
    model_dir = variant_dir / 'models'
    pred_dir = variant_dir / 'predictions'
    table_dir = variant_dir / 'tables'
    for p in [variant_dir, model_dir, pred_dir, table_dir]: p.mkdir(parents=True, exist_ok=True)

    ns = dict(globals())
    ns['CFG'] = _cfg_for_variant(variant)
    ns['FEATURE_NAMES'] = list(FEATURE_SETS[variant])
    ns['FEATURE_TABLES'] = FEATURE_TABLES
    ns['TRUTH'] = TRUTH
    ns['BUILDING_NAMES'] = BUILDING_NAMES
    ns['SEASONS'] = SEASONS
    ns['load_raw'] = load_raw
    ns['weather_raw'] = weather_raw
    ns['RADIATION_COL'] = RADIATION_COL
    ns['season_mask'] = season_mask
    ns['MODEL_DIR'] = model_dir
    ns['PRED_DIR'] = pred_dir
    ns['TABLE_DIR'] = table_dir
    ns['FIXED_EVENT_WEIGHTS'] = _fixed_weights_from_cfg(ns['CFG'])

    # Re-execute exact original operational-event semantics in this variant namespace.
    exec(SRC_EVENTS, ns)

    # Exact original model/scaler/sampling/loss/training schedule.
    exec(SRC_TRAIN_CORE, ns)
    exec(SRC_TRAIN_STAGE5, ns)

    # Variant-specific checkpoint identity only; it does not alter optimization.
    feature_sig = hashlib.sha256('|'.join(ns['FEATURE_NAMES']).encode()).hexdigest()[:12]
    weight_sig = hashlib.sha256(repr(ns['FIXED_EVENT_WEIGHTS']).encode()).hexdigest()[:12]
    config_sig = hashlib.sha256(repr(asdict(ns['CFG'])).encode()).hexdigest()[:12]
    source_sig = hashlib.sha256((SRC_EVENTS + SRC_FEATURES + SRC_TRAIN_CORE + SRC_TRAIN_STAGE5).encode()).hexdigest()[:12]
    ns['TRAINING_LOGIC_VERSION'] = (
        ns['TRAINING_LOGIC_VERSION']
        + f'__auditedV4__{variant}__F{feature_sig}__W{weight_sig}__C{config_sig}__S{source_sig}__D{TRAINING_DEPENDENCY_SIGNATURE[:12]}'
    )

    # Exact original held-out decoder/evaluation utilities.
    exec(SRC_EVAL_UTILS, ns)
    return ns

# All feature ablations intentionally use the SAME frozen feature-valid sample
# universe as the original paper. Thus differences arise from removed predictor
# columns (or event weights), not from different coverage/preprocessing rows.
_sample_universe_rows = []
for b in BUILDING_NAMES:
    base_idx = FEATURE_TABLES[b].index
    for variant, cols in FEATURE_SETS.items():
        finite = np.isfinite(FEATURE_TABLES[b][cols].to_numpy(float)).all()
        _sample_universe_rows.append({
            'building': b,
            'variant': variant,
            'rows': len(base_idx),
            'same_frozen_row_universe': True,
            'selected_features_all_finite': bool(finite),
        })
SAMPLE_UNIVERSE_AUDIT = pd.DataFrame(_sample_universe_rows)
assert SAMPLE_UNIVERSE_AUDIT[['same_frozen_row_universe','selected_features_all_finite']].all().all()
SAMPLE_UNIVERSE_AUDIT.to_csv(AUDIT_DIR / 'Ablation_Same_Sample_Universe_Audit.csv', index=False)

# Contract table showing that only the intended dimensions change.
contract_rows = []
for variant in FEATURE_SETS:
    c = CFG_BASE if variant == 'full_hvg_eventaware' else _cfg_for_variant(variant)
    contract_rows.append({
        'variant':variant,
        'n_features':len(FEATURE_SETS[variant]),
        'feature_names':' | '.join(FEATURE_SETS[variant]),
        'architecture':'64-32-16 + LayerNorm + tanh + dropout 0.05 + sigmoid',
        'seeds':str(tuple(c.lobo_seeds)),
        'batch_size':c.batch_size,
        'eval_batch_size':c.eval_batch_size,
        'weight_decay':c.weight_decay,
        'grad_clip_norm':c.grad_clip_norm,
        'curriculum_lrs':str(tuple(row[0] for row in c.curriculum)),
        'event_weights':str(_fixed_weights_from_cfg(c)),
        'decision_threshold':c.decision_threshold,
    })
ABLATION_CONTRACT = pd.DataFrame(contract_rows)
ABLATION_CONTRACT.to_csv(AUDIT_DIR / 'Ablation_Contract.csv', index=False)
display(ABLATION_CONTRACT[['variant','n_features','event_weights','curriculum_lrs']])
# Dependency smoke test for the exact original monthly held-out evaluator.
_smoke_ns = make_exact_variant_namespace('load_only_eventaware')
_smoke_feat = FEATURE_TABLES[BUILDING_NAMES[0]]
_sy, _sm = int(_smoke_feat.iloc[0]['year']), int(_smoke_feat.iloc[0]['month'])
_smoke_month = _smoke_feat.loc[(_smoke_feat['year']==_sy) & (_smoke_feat['month']==_sm)]
_smoke_qc = _smoke_ns['month_qc'](_smoke_month, TRUTH[BUILDING_NAMES[0]]['work']['binary_on'], _sy, _sm)
assert 'feature_interval_coverage' in _smoke_qc and 'truth_interval_coverage' in _smoke_qc
print('PASS: exact original held-out monthly evaluator dependencies (load_raw, season_mask, SEASONS) are present.')

# Recompute every canonical LOSO split and fold-local RobustScaler with the exact
# original FULL-feature training functions, then compare against the scaler saved in
# the frozen checkpoints.
_canon_scaler_ns = dict(globals())
_canon_scaler_ns['CFG'] = CFG_BASE
_canon_scaler_ns['FEATURE_NAMES'] = FEATURE_NAMES_FULL
_canon_scaler_ns['FEATURE_TABLES'] = FEATURE_TABLES
_canon_scaler_ns['TRUTH'] = TRUTH
_canon_scaler_ns['BUILDING_NAMES'] = BUILDING_NAMES
_canon_scaler_ns['FIXED_EVENT_WEIGHTS'] = _fixed_weights_from_cfg(CFG_BASE)
exec(SRC_TRAIN_CORE, _canon_scaler_ns)
fold_scaler_rows = []
for _test in BUILDING_NAMES:
    _train, _val = _canon_scaler_ns['fold_split'](_test)
    _train_df = _canon_scaler_ns['supervised_rows'](_train)
    _center, _scale = _canon_scaler_ns['fit_fold_scaler'](_train_df)
    _payload = torch.load(MODEL_DIR_CANONICAL / _test / f'seed_{CANONICAL_SEEDS[0]}_best.pt', map_location='cpu', weights_only=False)
    _center_ok = np.allclose(np.asarray(_payload['scaler_center'],float), np.asarray(_center,float), rtol=0, atol=1e-12)
    _scale_ok = np.allclose(np.asarray(_payload['scaler_scale'],float), np.asarray(_scale,float), rtol=0, atol=1e-12)
    _split_ok = (_payload['val_building']==_val and list(_payload['train_buildings'])==list(_train))
    fold_scaler_rows.append({'test_system':_test,'validation_system':_val,'split_exact':_split_ok,'scaler_center_exact':_center_ok,'scaler_scale_exact':_scale_ok})
FOLD_SCALER_AUDIT = pd.DataFrame(fold_scaler_rows)
assert FOLD_SCALER_AUDIT[['split_exact','scaler_center_exact','scaler_scale_exact']].all().all(), FOLD_SCALER_AUDIT
FOLD_SCALER_AUDIT.to_csv(AUDIT_DIR / 'Canonical_Fold_Scaler_Audit.csv', index=False)
print('PASS: canonical LOSO splits and fold-local RobustScaler parameters reproduce all frozen checkpoints exactly.')


In [ ]:
# CELL 7 — Train/resume the three non-canonical ablations
# Heavy cell: 3 variants × 7 LOSO folds × 5 seeds = 105 trained seed models.
RUN_ABLATION_TRAINING = True

ABLATION_NAMESPACES = {}
for variant in ['load_only_eventaware','load_weather_eventaware','full_no_event_weighting']:
    ns = make_exact_variant_namespace(variant)
    ABLATION_NAMESPACES[variant] = ns
    if RUN_ABLATION_TRAINING:
        print('\n' + '='*100)
        print('ABLATION:', variant)
        print('='*100)
        for test_system in BUILDING_NAMES:
            ns['train_fold_parallel_seeds'](test_system)
        gc.collect()
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    else:
        print('Training skipped for', variant, '— compatible checkpoints must already exist.')

# Comprehensive post-training audit before any ablation is evaluated.
ablation_checkpoint_rows = []
for variant in ['load_only_eventaware','load_weather_eventaware','full_no_event_weighting']:
    ns = ABLATION_NAMESPACES[variant]
    cfg_v = ns['CFG']
    expected_cfg = asdict(cfg_v)
    expected_features = list(FEATURE_SETS[variant])
    expected_version = ns['TRAINING_LOGIC_VERSION']
    for test_system in BUILDING_NAMES:
        i = BUILDING_NAMES.index(test_system)
        expected_val = BUILDING_NAMES[(i+1) % len(BUILDING_NAMES)]
        expected_train = [x for x in BUILDING_NAMES if x not in {test_system, expected_val}]
        for seed in CANONICAL_SEEDS:
            model_path = ns['MODEL_DIR'] / test_system / f'seed_{seed}_best.pt'
            hist_path = ns['MODEL_DIR'] / test_system / f'seed_{seed}_history.csv'
            meta_path = ns['MODEL_DIR'] / test_system / f'seed_{seed}_meta.json'
            assert model_path.exists() and hist_path.exists() and meta_path.exists(), (variant,test_system,seed)
            payload = torch.load(model_path, map_location='cpu', weights_only=False)
            hist = pd.read_csv(hist_path)
            schedule_ok = True
            stages_ok = set(pd.to_numeric(hist['stage']).astype(int).unique()) == set(range(1,len(cfg_v.curriculum)+1))
            min_epochs_ok = True
            warmup_caps_ok = True
            for stage_no, row in enumerate(cfg_v.curriculum, start=1):
                hs = hist.loc[pd.to_numeric(hist['stage']).astype(int).eq(stage_no)]
                if hs.empty:
                    schedule_ok = False; min_epochs_ok = False; continue
                schedule_ok &= bool(np.allclose(pd.to_numeric(hs['lr']), float(row[0]), rtol=0, atol=1e-15))
                for c,v in zip(_HISTORY_WEIGHT_COLS,row[1:]):
                    schedule_ok &= bool(np.allclose(pd.to_numeric(hs[c]), float(v), rtol=0, atol=1e-12))
                min_epochs_ok &= int(pd.to_numeric(hs['epochs_in_stage']).max()) >= int(cfg_v.min_epochs_per_stage)
                if stage_no < len(cfg_v.curriculum):
                    warmup_caps_ok &= int(pd.to_numeric(hs['epochs_in_stage']).max()) <= int(cfg_v.max_epochs_per_stage)
            row_a = {
                'variant':variant,'test_system':test_system,'seed':int(seed),
                'config_exact':payload['config']==expected_cfg,
                'feature_order_exact':list(payload['feature_names'])==expected_features,
                'building_set_exact':list(payload['building_set'])==BUILDING_NAMES,
                'test_system_exact':payload['test_building']==test_system,
                'validation_system_exact':payload['val_building']==expected_val,
                'training_systems_exact':list(payload['train_buildings'])==expected_train,
                'stage5_checkpoint':int(payload['best_stage'])==len(cfg_v.curriculum),
                'all_stages_reached':bool(payload['all_stages_reached']),
                'training_logic_version_exact':payload['training_logic_version']==expected_version,
                'history_all_stages_present':bool(stages_ok),
                'history_lr_weights_exact':bool(schedule_ok),
                'history_min_stage_epochs_ok':bool(min_epochs_ok),
                'history_warmup_caps_ok':bool(warmup_caps_ok),
                'history_total_epoch_ceiling_ok':int(pd.to_numeric(hist['epoch']).max()) <= int(cfg_v.max_epochs),
            }
            ablation_checkpoint_rows.append(row_a)
ABLATION_CHECKPOINT_AUDIT = pd.DataFrame(ablation_checkpoint_rows)
_bool_cols = [c for c in ABLATION_CHECKPOINT_AUDIT.columns if c not in {'variant','test_system','seed'}]
assert len(ABLATION_CHECKPOINT_AUDIT)==3*7*5
assert ABLATION_CHECKPOINT_AUDIT[_bool_cols].all().all(), ABLATION_CHECKPOINT_AUDIT.loc[~ABLATION_CHECKPOINT_AUDIT[_bool_cols].all(axis=1)]
ABLATION_CHECKPOINT_AUDIT.to_csv(AUDIT_DIR / 'Ablation_PostTraining_Checkpoint_and_History_Audit.csv', index=False)
print('PASS: all 105 non-canonical ablation checkpoints obey the exact requested split/config/stage schedule.')


In [ ]:
# CELL 8 — Exact original held-out inference/evaluation for each trained ablation
# No custom decoder is used here: the original paper's held-out inference cell is executed.
ABLATION_INTERVAL = {'full_hvg_eventaware': {b:INTERVAL_CANONICAL[b].copy() for b in BUILDING_NAMES}}
ABLATION_DAILY = [DAILY_CANONICAL.assign(variant='full_hvg_eventaware')]
ABLATION_POINTWISE = [POINTWISE_CANONICAL.assign(variant='full_hvg_eventaware')]

for variant in ['load_only_eventaware','load_weather_eventaware','full_no_event_weighting']:
    ns = ABLATION_NAMESPACES.get(variant) or make_exact_variant_namespace(variant)
    # Executes the exact original five-seed averaging (ddof=1 score SD), majority-vote
    # agreement diagnostic, fixed 0.5 threshold, isolated-flip cleanup, daily-event
    # extraction, pointwise metrics, and monthly evaluation.
    exec(SRC_HELDOUT_INFERENCE, ns)

    interval = {b:ns['INTERVAL_OUTPUTS'][b].copy() for b in BUILDING_NAMES}
    daily = ns['daily_schedule_df'].copy(); daily['variant'] = variant
    point = ns['pointwise_df'].copy(); point['variant'] = variant

    ABLATION_INTERVAL[variant] = interval
    ABLATION_DAILY.append(daily)
    ABLATION_POINTWISE.append(point)

ABLATION_DAILY_DF = pd.concat(ABLATION_DAILY, ignore_index=True)
ABLATION_POINTWISE_DF = pd.concat(ABLATION_POINTWISE, ignore_index=True)
ABLATION_DAILY_DF.to_csv(ABLATION_DIR / 'Ablation_Daily_Operational.csv', index=False)
ABLATION_POINTWISE_DF.to_csv(ABLATION_DIR / 'Ablation_Pointwise_BySystem.csv', index=False)

print('PASS: ablation decoding/evaluation used the exact original held-out inference cell.')

In [ ]:
# CELL 9 — Macro/pooled ablation metrics, CIs, and paired deltas vs full model
ABLATION_CLUSTERS = {}
ablation_rows = []
ablation_ci_rows = []
ablation_rep = {}

for variant in FEATURE_SETS:
    daily_v = ABLATION_DAILY_DF.loc[ABLATION_DAILY_DF['variant'].eq(variant)].copy()
    clusters = build_day_cluster_tables(ABLATION_INTERVAL[variant], daily_v)
    ABLATION_CLUSTERS[variant] = clusters
    central = compute_metrics_from_sampled_clusters([clusters[b] for b in BUILDING_NAMES])
    ablation_rows.append({'variant':variant,'n_features':len(FEATURE_SETS[variant]),**central})
    ci, reps = hierarchical_bootstrap(clusters, n_boot=BOOTSTRAP_REPS, seed=BOOTSTRAP_SEED, ci=CI_LEVEL)
    ci.insert(0,'variant',variant)
    ablation_ci_rows.append(ci)
    ablation_rep[variant] = reps

ABLATION_SUMMARY = pd.DataFrame(ablation_rows)
ABLATION_CI = pd.concat(ablation_ci_rows, ignore_index=True)
ABLATION_SUMMARY.to_csv(ABLATION_DIR / 'Ablation_Summary_Macro_and_Pooled.csv', index=False)
ABLATION_CI.to_csv(ABLATION_DIR / 'Ablation_Metrics_95CI.csv', index=False)

# Because every variant uses the same feature-valid rows/day universe and the same
# bootstrap seed, replicate r uses the same hierarchical resample. This permits
# paired uncertainty on each ablation-minus-full difference.
# Verify that every variant has the identical system/day cluster universe before
# pairing bootstrap replicate r across variants.
for variant, clusters in ABLATION_CLUSTERS.items():
    for b in BUILDING_NAMES:
        ref_dates = pd.to_datetime(ABLATION_CLUSTERS['full_hvg_eventaware'][b]['date']).reset_index(drop=True)
        var_dates = pd.to_datetime(clusters[b]['date']).reset_index(drop=True)
        assert ref_dates.equals(var_dates), (variant,b,'cluster day universe differs')

full_rep = ablation_rep['full_hvg_eventaware']
paired_rows = []
for variant in ['load_only_eventaware','load_weather_eventaware','full_no_event_weighting']:
    vr = ablation_rep[variant]
    assert vr.shape == full_rep.shape and list(vr.columns) == list(full_rep.columns)
    for metric in vr.columns:
        d = pd.to_numeric(vr[metric],errors='coerce') - pd.to_numeric(full_rep[metric],errors='coerce')
        d = d.dropna()
        central_delta = (
            ABLATION_SUMMARY.set_index('variant').loc[variant,metric]
            - ABLATION_SUMMARY.set_index('variant').loc['full_hvg_eventaware',metric]
        )
        paired_rows.append({
            'variant':variant,
            'reference_variant':'full_hvg_eventaware',
            'metric':metric,
            'delta_estimate':central_delta,
            'ci_low':d.quantile(0.025) if len(d) else np.nan,
            'ci_high':d.quantile(0.975) if len(d) else np.nan,
            'bootstrap_replicates_valid':len(d),
        })
PAIRED_DELTAS = pd.DataFrame(paired_rows)
PAIRED_DELTAS.to_csv(ABLATION_DIR / 'Ablation_Paired_Delta_vs_Full_95CI.csv', index=False)

display(ABLATION_SUMMARY)

## Reference-definition sensitivity

Alternative reference thresholds and formulations test how the constructed electrical ON/OFF label affects agreement and frozen-model scores. This is a sensitivity analysis, not validation against independent building-automation-system commands.

In [ ]:
# CELL 10 — Explicit independent-BAS-state audit across all supplied source workbooks
LOAD_FILE = TASK_DIR / CFG_BASE.load_file
WEATHER_FILE = TASK_DIR / CFG_BASE.weather_file
state_tokens = ('state','status','command','cmd','enable','on_off','onoff','running','run_signal','compressor_state')

def _workbook_header_audit(path):
    xls = pd.ExcelFile(path)
    rows = []
    time_series_candidates = []
    for sheet in xls.sheet_names:
        hdr = pd.read_excel(path, sheet_name=sheet, nrows=2)
        cols = [str(c) for c in hdr.columns]
        dt_cols = [c for c in cols if 'datetime' in c.lower() or 'timestamp' in c.lower() or c.lower() in {'date','time'}]
        token_hits = [c for c in cols if any(t in c.lower() for t in state_tokens)]
        is_time_series_sheet = bool(dt_cols)
        if is_time_series_sheet:
            # Power channels are not independent state evidence.
            non_power_hits = [
                c for c in token_hits
                if not c.lower().startswith('total_load_')
                and not c.lower().startswith('chiller_load_')
            ]
            for c in non_power_hits:
                time_series_candidates.append((sheet,c))
        rows.append({
            'workbook':Path(path).name,
            'sheet':sheet,
            'n_columns':len(cols),
            'datetime_like_columns':' | '.join(dt_cols),
            'state_token_columns':' | '.join(token_hits),
            'time_series_sheet':is_time_series_sheet,
        })
    return rows, time_series_candidates

bas_rows = []
bas_candidates = []
for wb in [LOAD_FILE, WEATHER_FILE]:
    rows, candidates = _workbook_header_audit(wb)
    bas_rows.extend(rows)
    bas_candidates.extend([(Path(wb).name,s,c) for s,c in candidates])

BAS_WORKBOOK_AUDIT = pd.DataFrame(bas_rows)
BAS_WORKBOOK_AUDIT.to_csv(SENSITIVITY_DIR / 'Independent_BAS_State_Workbook_Scan.csv', index=False)
BAS_STATE_AVAILABLE = len(bas_candidates) > 0
bas_audit = pd.DataFrame([{
    'supplied_workbooks':f'{LOAD_FILE.name} | {WEATHER_FILE.name}',
    'candidate_independent_bas_state_channels':' | '.join(f'{w}:{s}:{c}' for w,s,c in bas_candidates),
    'independent_bas_state_available':BAS_STATE_AVAILABLE,
    'selected_validation_path':'independent BAS state' if BAS_STATE_AVAILABLE else 'reference threshold/formulation sensitivity',
    'note':'Metadata columns such as formula_documentation.status are not time-series BAS state signals.',
}])
bas_audit.to_csv(SENSITIVITY_DIR / 'Independent_BAS_State_Audit.csv', index=False)
display(BAS_WORKBOOK_AUDIT)
display(bas_audit)

if BAS_STATE_AVAILABLE:
    raise RuntimeError(
        'A possible time-series BAS state/command channel was detected. Do not infer its semantics automatically; '
        'map and verify that channel explicitly before using it as independent ground truth.'
    )


In [ ]:
# CELL 11 — Alternative references using the exact original classifier/smoother
REFERENCE_VARIANTS = {}

for b in BUILDING_NAMES:
    w = TRUTH[b]['work'].copy()
    cfg0 = dict(BUILDINGS[b])
    chiller_col = cfg0['chiller']
    seed_median = float(TRUTH[b]['summary']['active_seed_meta']['seed_pool_median_kw'])

    variants = {
        'thr_0.15': {'ref':w['active_reference_kw'], 'aux_max_ratio':0.15},
        'thr_0.20_baseline': {'ref':w['active_reference_kw'], 'aux_max_ratio':0.20},
        'thr_0.25': {'ref':w['active_reference_kw'], 'aux_max_ratio':0.25},
        'thr_0.30': {'ref':w['active_reference_kw'], 'aux_max_ratio':0.30},
        'sttr_unsmoothed': {'ref':w['active_reference_raw_kw'], 'aux_max_ratio':0.20},
        'constant_seed_median': {'ref':pd.Series(seed_median,index=w.index), 'aux_max_ratio':0.20},
    }

    REFERENCE_VARIANTS[b] = {}
    for name, spec in variants.items():
        tmp = w.copy()
        tmp['active_reference_kw'] = pd.to_numeric(spec['ref'], errors='coerce').reindex(tmp.index)
        cfg = dict(cfg0)
        cfg['aux_max_ratio'] = float(spec['aux_max_ratio'])
        classified = classify_reference_states(tmp, cfg, chiller_col)
        binary = classified['reference_state'].map(STATE_TO_BINARY).astype(float)
        REFERENCE_VARIANTS[b][name] = binary

    # Strong equality assertion: values AND missing-value mask must match exactly.
    baseline = REFERENCE_VARIANTS[b]['thr_0.20_baseline']
    frozen = pd.to_numeric(w['binary_on'], errors='coerce')
    assert baseline.index.equals(frozen.index)
    assert np.array_equal(baseline.isna().to_numpy(), frozen.isna().to_numpy()), b
    m = baseline.notna()
    assert np.array_equal(baseline.loc[m].to_numpy(float), frozen.loc[m].to_numpy(float)), b

print('PASS: exact original classifier reproduces the frozen 0.20 binary reference for all seven systems.')

In [ ]:
# CELL 12 — Reference agreement and frozen-model robustness under each alternative reference
# Use exact original daily comparison semantics.
# SRC_EVAL_UTILS requires ScoreStatsMLP only for model-loading functions; define it by
# executing the original training core in a small canonical namespace.
_ref_ns = dict(globals())
_ref_ns['CFG'] = CFG_BASE
_ref_ns['FEATURE_NAMES'] = FEATURE_NAMES_FULL
_ref_ns['FEATURE_TABLES'] = FEATURE_TABLES
_ref_ns['BUILDING_NAMES'] = BUILDING_NAMES
_ref_ns['TRUTH'] = TRUTH
_ref_ns['SEASONS'] = SEASONS
_ref_ns['FIXED_EVENT_WEIGHTS'] = _fixed_weights_from_cfg(CFG_BASE)
exec(SRC_TRAIN_CORE, _ref_ns)
exec(SRC_EVENTS, _ref_ns)
exec(SRC_EVAL_UTILS, _ref_ns)
build_daily_comparison_exact = _ref_ns['build_daily_comparison']
event_detection_stats_exact = _ref_ns['event_detection_stats']
matched_timing_stats_exact = _ref_ns['matched_timing_stats']

agreement_rows = []
variant_interval = {}
variant_daily = []

variant_names = list(REFERENCE_VARIANTS[BUILDING_NAMES[0]].keys())
for variant in variant_names:
    variant_interval[variant] = {}
    for b in BUILDING_NAMES:
        base = REFERENCE_VARIANTS[b]['thr_0.20_baseline']
        ref = REFERENCE_VARIANTS[b][variant]
        common = base.notna() & ref.notna()
        y0 = base.loc[common].astype(int).to_numpy()
        y1 = ref.loc[common].astype(int).to_numpy()
        agreement_rows.append({
            'building':b,
            'reference_variant':variant,
            'baseline_valid_intervals':int(base.notna().sum()),
            'variant_valid_intervals':int(ref.notna().sum()),
            'n_common_intervals':int(len(y0)),
            'valid_mask_agreement':float(np.mean(base.notna().to_numpy()==ref.notna().to_numpy())),
            'interval_agreement_vs_baseline':float(np.mean(y0==y1)) if len(y0) else np.nan,
            'f1_vs_baseline':float(f1_score(y0,y1,zero_division=0)) if len(y0) else np.nan,
            'mcc_vs_baseline':float(matthews_corrcoef(y0,y1)) if len(y0) and len(np.unique(y0))>1 and len(np.unique(y1))>1 else np.nan,
            'on_fraction':float(np.mean(y1)) if len(y1) else np.nan,
            'baseline_on_fraction':float(np.mean(y0)) if len(y0) else np.nan,
        })

        pred = INTERVAL_CANONICAL[b].copy()
        pred['true_state'] = ref.reindex(pred.index)
        variant_interval[variant][b] = pred

        daily = build_daily_comparison_exact(ref, pd.to_numeric(pred['predicted_state'], errors='coerce'))
        daily['building'] = b
        daily['variant'] = variant
        variant_daily.append(daily)

REFERENCE_AGREEMENT = pd.DataFrame(agreement_rows)
REFERENCE_DAILY = pd.concat(variant_daily, ignore_index=True)
REFERENCE_AGREEMENT.to_csv(SENSITIVITY_DIR / 'Reference_Sensitivity_Agreement_BySystem.csv', index=False)
REFERENCE_DAILY.to_csv(SENSITIVITY_DIR / 'Reference_Sensitivity_Daily_Operational.csv', index=False)

summary_rows = []
for variant in variant_names:
    daily_v = REFERENCE_DAILY.loc[REFERENCE_DAILY['variant'].eq(variant)].copy()
    clusters = build_day_cluster_tables(variant_interval[variant], daily_v)
    summary_rows.append({'reference_variant':variant, **compute_metrics_from_sampled_clusters([clusters[b] for b in BUILDING_NAMES])})
REFERENCE_SUMMARY = pd.DataFrame(summary_rows)
REFERENCE_SUMMARY.to_csv(SENSITIVITY_DIR / 'Reference_Sensitivity_Macro_and_Pooled.csv', index=False)

# Direct construct-sensitivity of the REFERENCE itself: compare each alternative
# reference with the canonical reference using the exact original daily-event rules.
# This separates label-definition shifts from model prediction error.
direct_ref_rows = []
for variant in variant_names:
    for b in BUILDING_NAMES:
        baseline = REFERENCE_VARIANTS[b]['thr_0.20_baseline']
        alternative = REFERENCE_VARIANTS[b][variant]
        comp = build_daily_comparison_exact(baseline, alternative)
        elig = comp['daily_eval_eligible'].fillna(False).to_numpy(bool)
        row = {'building':b,'reference_variant':variant,'eligible_days':int(elig.sum())}
        for ev,col in [('on','turn_on'),('off','turn_off')]:
            det = event_detection_stats_exact(
                comp[f'true_{col}'], comp[f'pred_{col}'], f'{ev}', eval_mask=elig
            )
            tm = matched_timing_stats_exact(
                comp[f'true_{col}'], comp[f'pred_{col}'], f'{ev}', eval_mask=elig
            )
            row.update({
                f'{ev}_event_precision_vs_baseline':det[f'{ev}_precision'],
                f'{ev}_event_recall_vs_baseline':det[f'{ev}_recall'],
                f'{ev}_event_f1_vs_baseline':det[f'{ev}_f1'],
                f'{ev}_matched_n_vs_baseline':tm[f'{ev}_matched_n'],
                f'{ev}_timing_mae_vs_baseline_min':tm[f'{ev}_mae_min'],
                f'{ev}_timing_median_ae_vs_baseline_min':tm[f'{ev}_median_ae_min'],
                f'{ev}_timing_q90_ae_vs_baseline_min':tm[f'{ev}_q90_ae_min'],
            })
        direct_ref_rows.append(row)
DIRECT_REFERENCE_SENSITIVITY = pd.DataFrame(direct_ref_rows)
DIRECT_REFERENCE_SENSITIVITY.to_csv(SENSITIVITY_DIR / 'Reference_Sensitivity_Direct_Event_Shifts_BySystem.csv', index=False)

# Uncertainty bounds for frozen-model performance under each alternative reference.
reference_ci_rows = []
for j, variant in enumerate(variant_names):
    daily_v = REFERENCE_DAILY.loc[REFERENCE_DAILY['variant'].eq(variant)].copy()
    clusters = build_day_cluster_tables(variant_interval[variant], daily_v)
    ci, _ = hierarchical_bootstrap(
        clusters,
        n_boot=BOOTSTRAP_REPS,
        seed=BOOTSTRAP_SEED,  # same resample plan for paired comparability
        ci=CI_LEVEL,
    )
    ci.insert(0,'reference_variant',variant)
    reference_ci_rows.append(ci)
REFERENCE_CI = pd.concat(reference_ci_rows, ignore_index=True)
REFERENCE_CI.to_csv(SENSITIVITY_DIR / 'Reference_Sensitivity_Macro_Pooled_95CI.csv', index=False)

display(REFERENCE_SUMMARY)
display(DIRECT_REFERENCE_SENSITIVITY)


In [ ]:
# CELL 13 — Final validation-extension manifest
manifest = {
    'source_notebook':str(ORIGINAL_NOTEBOOK),
    'canonical_result_dir':str(CANONICAL_RESULT_DIR),
    'validation_dir':str(VALIDATION_DIR),
    'fidelity':{
        'original_config_extracted_from_source':True,
        'all_35_canonical_checkpoint_configs_exact':True,
        'frozen_preprocessed_feature_tables_used_directly':True,
        'raw_excel_inputs_match_frozen_truth_exactly':True,
        'canonical_training_history_schedule_audited':True,
        'frozen_data_signature':FROZEN_DATA_SIGNATURE,
        'training_dependency_signature':TRAINING_DEPENDENCY_SIGNATURE,
        'original_raw_data_and_season_logic_executed':True,
        'canonical_metric_engine_parity_checked':True,
        'canonical_loso_split_and_scaler_recomputed_exactly':True,
        'all_105_ablation_checkpoints_post_audited':True,
        'training_core_source_sha256':hashlib.sha256(SRC_TRAIN_CORE.encode()).hexdigest(),
        'training_stage5_source_sha256':hashlib.sha256(SRC_TRAIN_STAGE5.encode()).hexdigest(),
        'heldout_inference_source_sha256':hashlib.sha256(SRC_HELDOUT_INFERENCE.encode()).hexdigest(),
    },
    'building_set':BUILDING_NAMES,
    'canonical_seeds':list(CANONICAL_SEEDS),
    'canonical_config':asdict(CFG_BASE),
    'feature_sets':FEATURE_SETS,
    'bootstrap':{
        'replicates':BOOTSTRAP_REPS,
        'seed':BOOTSTRAP_SEED,
        'ci_level':CI_LEVEL,
        'scheme':'systems with replacement, then calendar-day clusters with replacement within each sampled system copy',
    },
    'reference_sensitivity':{
        'independent_bas_state_available':BAS_STATE_AVAILABLE,
        'variants':variant_names,
        'baseline_exactly_reproduced':True,
        'note':'Sensitivity tests robustness of the constructed reference; it is not independent physical-state validation.',
    },
    'key_outputs':{
        'fidelity_checkpoint_audit':str(AUDIT_DIR/'Canonical_Checkpoint_Audit.csv'),
        'fidelity_preprocessing_audit':str(AUDIT_DIR/'Frozen_Preprocessing_Audit.csv'),
        'canonical_bootstrap_ci':str(BOOTSTRAP_DIR/'Bootstrap_Primary_Metrics_Macro_Pooled_95CI.csv'),
        'ablation_summary':str(ABLATION_DIR/'Ablation_Summary_Macro_and_Pooled.csv'),
        'ablation_ci':str(ABLATION_DIR/'Ablation_Metrics_95CI.csv'),
        'ablation_paired_delta_ci':str(ABLATION_DIR/'Ablation_Paired_Delta_vs_Full_95CI.csv'),
        'reference_agreement':str(SENSITIVITY_DIR/'Reference_Sensitivity_Agreement_BySystem.csv'),
        'reference_summary':str(SENSITIVITY_DIR/'Reference_Sensitivity_Macro_and_Pooled.csv'),
        'reference_direct_event_shifts':str(SENSITIVITY_DIR/'Reference_Sensitivity_Direct_Event_Shifts_BySystem.csv'),
        'reference_ci':str(SENSITIVITY_DIR/'Reference_Sensitivity_Macro_Pooled_95CI.csv'),
    },
}
with open(VALIDATION_DIR / 'Validation_Extensions_AuditedV4_Manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, default=str)

print('Done. Validation outputs written to:')
print(VALIDATION_DIR)